<a href="https://colab.research.google.com/github/mahieshwar-budati/Basic-Advance-RAG/blob/main/ad_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Everything

In [3]:
!pip install -q \
pymilvus \
milvus-lite \
sentence-transformers==2.7.0 \
transformers==4.41.2 \
langchain \
langchain-community \
langchain-text-splitters \
unstructured \
pypdf
!pip install requests==2.32.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured 0.21.5 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [4]:
!pip install "unstructured[pdf]"

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [9]:
# ---------- Imports ----------
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.vectorstores import Milvus
from pymilvus import connections

# ---------- Load PDF (Unstructured) ----------
file_path = "/content/be.pdf"
all_text = []

loader = UnstructuredPDFLoader(file_path)
documents = loader.load()

for doc in documents:
    all_text.append(doc.page_content)

print("PDF loaded")


# ---------- Chunking ----------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

full_text = "\n".join(all_text)
chunks = splitter.split_text(full_text)

print("Chunks created:", len(chunks))


# ---------- Embedding Model ----------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


PDF loaded
Chunks created: 3


/tmp/ipython-input-2300/1327012550.py:36: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Connect First
from pymilvus import connections

connections.connect(
    alias="default",
    uri="./milvus_demo.db"
)

print("Milvus connected 🚀")

# Drop Collection If It Exists
# Milvus will crash if collection already exists.

from pymilvus import utility

if utility.has_collection("rag_collection"):
    utility.drop_collection("rag_collection")
    print("Old collection dropped")

# Now Create Collection (Final Working Block)

from pymilvus import Collection, CollectionSchema, FieldSchema, DataType

schema = CollectionSchema([
    FieldSchema("id", DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=384),
])

collection = Collection("rag_collection", schema)

print("Collection ready ✅")

Milvus connected 🚀
Collection ready ✅


In [ ]:
# ---------- Imports ----------
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ---------- Load FLAN-T5 ----------
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model_llm = T5ForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_llm = model_llm.to(device)

print("FLAN-T5 loaded on:", device)


# ---------- Query Loop ----------
print("\nRAG Chatbot Ready! Type 'exit' to quit.\n")

while True:
    try:
        query = input("Ask question: ").strip()

        if query.lower() == "exit":
            print("Goodbye!")
            break

        if not query:
            print("Please enter a valid question.")
            continue

        # ---------- Retrieve context from Milvus ----------
        retrieved_chunks = search_milvus(query, top_k=3)

        if not retrieved_chunks:
            print("\nNo relevant documents found.\n")
            continue

        context = "\n".join([chunk[:500] for chunk in retrieved_chunks])

        # ---------- Prompt ----------
        prompt = f"""
Use the provided context to answer the question clearly.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

        # ---------- Tokenize ----------
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        # ---------- Generate ----------
        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.5,
            do_sample=False,
            top_p=0.5,
            repetition_penalty=1.1
        )

        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print("\nAnswer:\n", answer)
        print("-" * 60)

    except KeyboardInterrupt:
        print("\nSession ended.")
        break

    except Exception as e:
        print("\nError:", e)
        continue

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded on: cpu

RAG Chatbot Ready! Type 'exit' to quit.

